# Label Studio — Manual Labeling Workflow

This notebook replaces the circular model-generated labels with **your own domain knowledge**.

**Workflow:**
```
1. Run this notebook  →  produces tasks.json
2. Import into Label Studio  →  label bikes manually (with Finn.no photos open)
3. Export from Label Studio  →  produces annotations.json
4. Run Section 5 of this notebook  →  produces finn_labeled.csv
5. Use finn_labeled.csv in bike_classifier_training.ipynb
```

**Pre-annotation:** The price regression model's prediction is pre-loaded into each task so you can just confirm or correct it — much faster than labeling from scratch.

## 0. Setup

### Install Label Studio (once)
```bash
pip install label-studio
```

### Start Label Studio
Run this in a terminal and leave it running while you label:
```bash
label-studio start
```
This opens `http://localhost:8080` in your browser. Create an account (local only) and you're ready.

In [21]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import r2_score
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings('ignore')

## 1. Load and preprocess data

In [22]:
df_raw = pd.read_csv('../data/finn_ad_data.csv', dtype=str)

# ── Price ──────────────────────────────────────────────────────────────────
df_raw['price_nok'] = pd.to_numeric(df_raw['price_nok'], errors='coerce')
df = df_raw.dropna(subset=['price_nok']).copy()
df = df[(df['price_nok'] >= 500) & (df['price_nok'] <= 250_000)].copy()

# ── Condition score ────────────────────────────────────────────────────────
CONDITION_MAP = {
    'Helt ny - Uåpnet/med lapp':                 5,
    'Som ny - Ikke synlig brukt':                4,
    'Pent brukt - I god stand':                  3,
    'Godt brukt - Synlig brukt':                 2,
    'Må fikses - Noen mangler/ødelagte deler':   1,
}
df['condition_score'] = df['tilstand'].map(CONDITION_MAP).fillna(3.0)

# ── Groupset tier ──────────────────────────────────────────────────────────
def groupset_tier(gs):
    if not isinstance(gs, str) or not gs.strip(): return 0
    g = gs.lower()
    if any(k in g for k in ['dura-ace','dura ace','duraace','sram red','super record','record']): return 5
    if any(k in g for k in ['ultegra','force','chorus','athena']): return 4
    if any(k in g for k in ['105','rival','centaur','veloce']): return 3
    if any(k in g for k in ['tiagra','apex','potenza','xenon']): return 2
    if any(k in g for k in ['sora','claris','tourney','acera','campagnolo']): return 1
    if any(k in g for k in ['shimano','sram','campagnolo']): return 2
    return 0

df['groupset_tier'] = df['groupset'].apply(groupset_tier)

# ── Brand tier ─────────────────────────────────────────────────────────────
BRAND_TIER_4 = {'pinarello','colnago','cervelo','bmc','wilier','look','time','de rosa','specialized'}
BRAND_TIER_3 = {'trek','cannondale','giant','canyon','bianchi','ridley','scott','felt','cube',
                'orbea','focus','lapierre','factor','rose','ktm','eddy merckx','cinelli','fuji'}
BRAND_TIER_2 = {'merida','btwin','decathlon','raleigh','norco','marin','nakamura','hardrocx','nor','nishiki'}

def brand_tier(b):
    if not isinstance(b, str) or not b.strip(): return 1
    bl = b.lower().strip()
    if bl in BRAND_TIER_4: return 4
    if bl in BRAND_TIER_3: return 3
    if bl in BRAND_TIER_2: return 2
    return 1

df['brand_tier'] = df['brand'].apply(brand_tier)

# ── Boolean features ───────────────────────────────────────────────────────
for col in ['carbon_frame','carbon_wheels','electronic_shifting']:
    df[col] = df[col].map({'true': 1, 'false': 0}).fillna(0).astype(int)

# ── Bike age ───────────────────────────────────────────────────────────────
CURRENT_YEAR = 2026
df['year_num'] = pd.to_numeric(df['year'], errors='coerce')
df.loc[(df['year_num'] < 1980) | (df['year_num'] > CURRENT_YEAR), 'year_num'] = np.nan
df['bike_age'] = (CURRENT_YEAR - df['year_num']).fillna(5.0)

# ── Frame size ─────────────────────────────────────────────────────────────
SIZE_MAP = {'xxs':48,'xs':50,'s':52,'small':52,'m':54,'medium':54,
            'l':56,'large':56,'xl':58,'xlarge':58,'x-large':58,'xxl':60}

def parse_framesize(val):
    if not isinstance(val, str) or not val.strip(): return np.nan
    v = val.lower().strip()
    m = re.search(r'\((\d{2,3})\)', v)
    if m: return float(m.group(1))
    m = re.match(r'^(\d{2,3})\s*cm?$', v)
    if m: return float(m.group(1))
    m = re.match(r'^(\d{2,3})$', v)
    if m: return float(m.group(1))
    for key, cm in SIZE_MAP.items():
        if v.startswith(key): return float(cm)
    return np.nan

df['framesize_cm'] = df['framesize'].apply(parse_framesize)
df.loc[(df['framesize_cm'] < 44) | (df['framesize_cm'] > 65), 'framesize_cm'] = np.nan
df['framesize_cm'] = df['framesize_cm'].fillna(df['framesize_cm'].median())

FEATURES = ['condition_score','groupset_tier','brand_tier',
            'carbon_frame','carbon_wheels','electronic_shifting',
            'bike_age','framesize_cm']

X = df[FEATURES].copy()
y = df['price_nok'].copy()

print(f'Ready: {len(df)} bikes')

Ready: 722 bikes


## 2. Price regression — generate pre-annotations

We fit XGBoost on all data (no CV needed here — we just want a reasonable pre-annotation, not leak-free labels).  
You will correct any wrong predictions during manual labeling.

In [23]:
y_log = np.log1p(y)
cv    = KFold(n_splits=5, shuffle=True, random_state=42)

xgb = XGBRegressor(
    n_estimators=300, learning_rate=0.05, max_depth=4,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1, verbosity=0,
)

# OOF so residuals are honest
xgb_log_pred = cross_val_predict(xgb, X, y_log, cv=cv)
df['price_predicted'] = np.expm1(xgb_log_pred)
df['price_residual_pct'] = (y - df['price_predicted']) / df['price_predicted'] * 100

GOOD_THRESHOLD = -20
BAD_THRESHOLD  = +20

def pre_label(pct):
    if pct < GOOD_THRESHOLD: return 'good_deal'
    if pct > BAD_THRESHOLD:  return 'bad_deal'
    return 'ok_deal'

df['model_label'] = df['price_residual_pct'].apply(pre_label)

print('Pre-annotation distribution:')
print(df['model_label'].value_counts())
print(f'\nOOF R²: {r2_score(y, df["price_predicted"]):.3f}')

Pre-annotation distribution:
model_label
ok_deal      274
bad_deal     225
good_deal    223
Name: count, dtype: int64

OOF R²: 0.557


## 3. Select batch

In [24]:
# ── Settings ───────────────────────────────────────────────────────────────
BATCH_SIZE   = 50    # bikes per batch
BATCH_NUMBER = 1     # increment this each time you run a new labeling round
RANDOM_SEED  = BATCH_NUMBER * 7  # different seed per batch for non-overlapping samples

# Track which bikes have already been exported (avoid re-labeling)
ALREADY_LABELED_PATH = Path('../data/already_labeled_ids.json')
if ALREADY_LABELED_PATH.exists():
    already_labeled = set(json.loads(ALREADY_LABELED_PATH.read_text()))
else:
    already_labeled = set()

pool = df[~df['ad_id'].isin(already_labeled)].copy()
print(f'Pool available: {len(pool)}  |  Already labeled: {len(already_labeled)}')

# Stratified sample: equal share of pre-annotated good/ok/bad
batch_parts = []
per_class = BATCH_SIZE // 3
for label in ['good_deal', 'ok_deal', 'bad_deal']:
    subset = pool[pool['model_label'] == label]
    n = min(per_class, len(subset))
    batch_parts.append(subset.sample(n=n, random_state=RANDOM_SEED))

batch = pd.concat(batch_parts).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
print(f'Batch size: {len(batch)}')
print(batch['model_label'].value_counts())

Pool available: 722  |  Already labeled: 0
Batch size: 48
model_label
good_deal    16
ok_deal      16
bad_deal     16
Name: count, dtype: int64


## 4. Build Label Studio tasks and export

In [25]:
def make_html(row):
    """Render a bike listing as HTML for display inside Label Studio."""
    carbon_tags = []
    if row['carbon_frame']:        carbon_tags.append('✓ Carbon frame')
    if row['carbon_wheels']:       carbon_tags.append('✓ Carbon wheels')
    if row['electronic_shifting']: carbon_tags.append('✓ Electronic shifting')
    carbon_str = ' &nbsp;|&nbsp; '.join(carbon_tags) if carbon_tags else '—'

    residual = row['price_residual_pct']
    residual_color = '#22c55e' if residual < -10 else ('#ef4444' if residual > 10 else '#6b7280')
    residual_str = f"{residual:+.1f}%"

    groupset = row['groupset'] if isinstance(row['groupset'], str) and row['groupset'] else '—'
    tilstand = row['tilstand'] if isinstance(row['tilstand'], str) and row['tilstand'] else '—'
    brand    = row['brand']    if isinstance(row['brand'],    str) and row['brand']    else '—'
    location = row['location'] if isinstance(row['location'], str) and row['location'] else '—'

    return f"""
<div style="font-family:sans-serif;max-width:680px;border:1px solid #e2e8f0;
            border-radius:10px;padding:20px;background:#fafafa">

  <h2 style="margin:0 0 4px;font-size:18px;color:#1e293b">{row['title']}</h2>
  <p style="margin:0 0 14px;color:#64748b;font-size:13px">{location}</p>

  <div style="display:flex;gap:24px;margin-bottom:14px">
    <div>
      <div style="font-size:11px;text-transform:uppercase;color:#94a3b8;letter-spacing:.5px">Asking price</div>
      <div style="font-size:22px;font-weight:700;color:#1e293b">{int(row['price_nok']):,} NOK</div>
    </div>
    <div>
      <div style="font-size:11px;text-transform:uppercase;color:#94a3b8;letter-spacing:.5px">Model expected</div>
      <div style="font-size:22px;font-weight:700;color:#1e293b">{int(row['price_predicted']):,} NOK</div>
    </div>
    <div>
      <div style="font-size:11px;text-transform:uppercase;color:#94a3b8;letter-spacing:.5px">Deviation</div>
      <div style="font-size:22px;font-weight:700;color:{residual_color}">{residual_str}</div>
    </div>
  </div>

  <table style="border-collapse:collapse;width:100%;font-size:14px;margin-bottom:16px">
    <tr style="background:#f1f5f9">
      <td style="padding:6px 10px;color:#64748b">Condition</td>
      <td style="padding:6px 10px;font-weight:600">{tilstand}</td>
      <td style="padding:6px 10px;color:#64748b">Brand</td>
      <td style="padding:6px 10px;font-weight:600">{brand}</td>
    </tr>
    <tr>
      <td style="padding:6px 10px;color:#64748b">Groupset</td>
      <td style="padding:6px 10px;font-weight:600">{groupset}</td>
      <td style="padding:6px 10px;color:#64748b">Frame size</td>
      <td style="padding:6px 10px;font-weight:600">{row['framesize'] if isinstance(row['framesize'], str) and row['framesize'] else '—'}</td>
    </tr>
    <tr style="background:#f1f5f9">
      <td style="padding:6px 10px;color:#64748b">Upgrades</td>
      <td colspan="3" style="padding:6px 10px;font-weight:600">{carbon_str}</td>
    </tr>
  </table>

  <a href="{row['source_url']}" target="_blank"
     style="display:inline-block;background:#0f172a;color:white;padding:9px 18px;
            border-radius:6px;text-decoration:none;font-size:14px;font-weight:600">
    🔗 Open on Finn.no (photos &amp; full description)
  </a>
</div>
"""


def build_task(row):
    """Build a Label Studio task dict with pre-annotation."""
    return {
        'data': {
            'ad_id':      row['ad_id'],
            'html':       make_html(row),
            'title':      row['title'],
            'price_nok':  int(row['price_nok']),
            'source_url': row['source_url'],
        },
        'predictions': [{
            'model_version': 'xgb_price_regression_v1',
            'score': max(0.0, 1.0 - abs(row['price_residual_pct']) / 100),
            'result': [{
                'id':         f"pred_{row['ad_id']}",
                'type':       'choices',
                'from_name':  'deal',
                'to_name':    'listing',
                'value':      {'choices': [row['model_label']]},
            }],
        }],
    }


tasks = [build_task(row) for _, row in batch.iterrows()]

OUT_PATH = Path(f'../data/label_studio_batch_{BATCH_NUMBER}.json')
OUT_PATH.write_text(json.dumps(tasks, ensure_ascii=False, indent=2), encoding='utf-8')                                                       
print(f'Exported {len(tasks)} tasks → {OUT_PATH}')

Exported 48 tasks → ..\data\label_studio_batch_1.json


## 5. Label Studio project setup

### 5.1 Labeling configuration

In Label Studio, create a new project and paste this XML into **"Labeling Setup" → "Custom template"**:

```xml
<View>
  <HyperText name="listing" value="$html" inline="true"/>
  <Choices name="deal" toName="listing" choice="single" showInLine="true"
           required="true" requiredMessage="Please select a deal quality">
    <Choice value="good_deal"
            style="background:#22c55e;color:white;font-weight:600;
                   padding:8px 20px;border-radius:6px;font-size:15px">
      Good deal
    </Choice>
    <Choice value="ok_deal"
            style="background:#f59e0b;color:white;font-weight:600;
                   padding:8px 20px;border-radius:6px;font-size:15px">
      OK deal
    </Choice>
    <Choice value="bad_deal"
            style="background:#ef4444;color:white;font-weight:600;
                   padding:8px 20px;border-radius:6px;font-size:15px">
      Bad deal
    </Choice>
  </Choices>
</View>
```

### 5.2 Import tasks

1. In your project, click **Import**
2. Upload `data/label_studio_batch_1.json`
3. The model's pre-annotation will be pre-selected for each bike
4. Click the listing link to view photos, then confirm or change the label
5. Press **Submit** (or `Ctrl+Enter`) to move to the next bike

### 5.3 Export

When done, click **Export** → **JSON** and save the file as `data/label_studio_export_batch_1.json`.

## 6. Load Label Studio export

Run this section after exporting from Label Studio.

In [27]:
EXPORT_PATH = Path(f'../data/label_studio_export_batch_{BATCH_NUMBER}.json')

if not EXPORT_PATH.exists():
    raise FileNotFoundError(
        f'{EXPORT_PATH} not found.\n'
        'Export from Label Studio first: Export → JSON, save to data/ folder.'
    )

with open(EXPORT_PATH, encoding='utf-8') as f:
    annotations = json.load(f)

print(f'Loaded {len(annotations)} annotations')

Loaded 48 annotations


In [28]:
def parse_annotation(task):
    """Extract ad_id and the human label from a Label Studio export task."""
    ad_id = task['data']['ad_id']

    # Prefer the most recent human annotation; fall back to prediction
    annots = task.get('annotations', [])
    human  = [a for a in annots if not a.get('was_cancelled', False)]

    if not human:
        # Unannotated — skip
        return None

    latest = sorted(human, key=lambda a: a.get('updated_at', ''))[-1]
    choices = latest['result'][0]['value']['choices']
    label   = choices[0] if choices else None

    return {'ad_id': ad_id, 'human_label': label}


records = [r for r in (parse_annotation(t) for t in annotations) if r]
labeled_df = pd.DataFrame(records)
print(f'Parsed labels: {len(labeled_df)}')
print(labeled_df['human_label'].value_counts())

Parsed labels: 47
human_label
Good deal    20
OK deal      18
Bad deal      9
Name: count, dtype: int64


In [29]:
# How often did you agree with the model?
agreement = batch.merge(labeled_df, on='ad_id', how='inner')
agreement['agreed'] = agreement['model_label'] == agreement['human_label']
print(f'Agreement rate: {agreement["agreed"].mean():.1%}')
print()
print(agreement.groupby(['model_label', 'human_label']).size().rename('count').to_string())

Agreement rate: 0.0%

model_label  human_label
bad_deal     Bad deal        7
             Good deal       2
             OK deal         6
good_deal    Good deal      10
             OK deal         6
ok_deal      Bad deal        2
             Good deal       8
             OK deal         6


## 7. Save labeled dataset and update ID tracker

In [30]:
# Merge human labels back onto the full feature set
new_labeled = df.merge(labeled_df, on='ad_id', how='inner').copy()
new_labeled = new_labeled.rename(columns={'human_label': 'deal_label'})

# Append to any previously labeled batches
FINAL_PATH = Path('../data/finn_labeled.csv')
if FINAL_PATH.exists():
    existing = pd.read_csv(FINAL_PATH, dtype={'ad_id': str})
    combined = pd.concat([existing, new_labeled], ignore_index=True)
    combined = combined.drop_duplicates(subset='ad_id', keep='last')
else:
    combined = new_labeled

combined.to_csv(FINAL_PATH, index=False)
print(f'Total labeled bikes saved: {len(combined)} → {FINAL_PATH}')
print(combined['deal_label'].value_counts())

# Update the ID tracker so next batch doesn't repeat these bikes
all_labeled_ids = list(set(combined['ad_id'].astype(str).tolist()))
ALREADY_LABELED_PATH.write_text(json.dumps(all_labeled_ids, indent=2))
print(f'\nID tracker updated: {len(all_labeled_ids)} bikes done')

Total labeled bikes saved: 47 → ..\data\finn_labeled.csv
deal_label
Good deal    20
OK deal      18
Bad deal      9
Name: count, dtype: int64

ID tracker updated: 47 bikes done


## 8. Next steps

- **More batches:** Increment `BATCH_NUMBER` in Section 3 and re-run from there to get the next 50 bikes
- **Train the classifier:** Use `finn_labeled.csv` in `bike_classifier_training.ipynb`
  - Replace `finn_preprocessed.csv` with `finn_labeled.csv`
  - Change `TARGET = 'deal_label'` (already correct)
  - You now have ground-truth labels from domain knowledge, not circular model predictions
- **Minimum recommended:** ~150–200 labeled bikes (3–4 batches) for a reliable classifier